# Relative Active Graph (RAG) — C4 Training & Streamlit UI

Trains the **MTLG lexicon** and **TRD bootstrapper** on a streaming subset of the
[C4 corpus](https://huggingface.co/datasets/allenai/c4), runs the integration test
suite, then launches the interactive **Streamlit UI** via a public ngrok tunnel.

The notebook **clones this repository directly** — all source is loaded from the
live codebase, never from inline copies.

| Step | Description |
|------|-------------|
| 1 | Clone repo from GitHub |
| 2 | Install Python dependencies |
| 3 | Download Stanza UD model |
| 4 | Configure demo parameters |
| 5 | Stream C4, parse → MTLG graphs |
| 6 | Induce MTLG lexicon (MLE) |
| 7 | Bootstrap TRD clusters (k-means) |
| 8 | Evaluate on held-out sentences |
| 9 | Run integration test suite |
| 10 | Visualise modal/UCCA distributions |
| 11 | Inference demo on new sentences |
| 12 | Render MTLG dependency graphs |
| 13 | Download trained artefacts |
| 14 | Launch Streamlit UI (public ngrok URL) |

## Step 1 — Clone repo from GitHub

In [ ]:
import os

REPO_URL  = "https://github.com/Eupham/Relative_Active_Graph.git"
REPO_DIR  = "/content/Relative_Active_Graph"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin master

# Confirm the branch / commit we are running against
!git -C {REPO_DIR} log --oneline -5

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/lcs/requirements.txt pyngrok

## Step 3 — Add repo modules to path & download Stanza model

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/lcs/induction")

import stanza

LANGUAGE = "en"  # change to any Stanza-supported language code
stanza.download(LANGUAGE, processors="tokenize,pos,lemma,depparse", verbose=False)
print("Stanza model ready.")

## Step 4 — Demo parameters

In [ ]:
TRAIN_SAMPLES  = 300   # sentences streamed from C4 for training
HELD_OUT       = 50    # sentences reserved for evaluation
N_TRD_CLUSTERS = 16   # k-means TRD clusters

print(f"Train samples : {TRAIN_SAMPLES}")
print(f"Held-out      : {HELD_OUT}")
print(f"TRD clusters  : {N_TRD_CLUSTERS}")

## Steps 5-8 — Stream C4, parse, induce lexicon, bootstrap TRDs, evaluate

In [ ]:
from mc4_stream import stream_mc4
from ud_parser import UdParser
from ud_to_mtlg import ud_tree_to_mtlg
from morphological_fst import preprocess_for_type_assignment
from mtlg_inducer import MtlgInducer
from trd_bootstrap import TrdBootstrapper, graph_to_modal_vector

parser = UdParser(LANGUAGE)
all_graphs = []
errors = 0

print(f"Streaming {TRAIN_SAMPLES + HELD_OUT} sentences from mC4 ({LANGUAGE})...")
for i, item in enumerate(stream_mc4(LANGUAGE, max_samples=TRAIN_SAMPLES + HELD_OUT)):
    try:
        tokens   = preprocess_for_type_assignment(item["text"], LANGUAGE)
        sentence = " ".join(t.split("[")[0] for t in tokens)
        tree     = parser.parse(sentence)
        graph    = ud_tree_to_mtlg(tree)
        if graph.nodes:
            all_graphs.append(graph)
    except Exception as e:
        errors += 1

    if (i + 1) % 50 == 0:
        print(f"  processed {i+1}/{TRAIN_SAMPLES + HELD_OUT} …")

train_graphs = all_graphs[:TRAIN_SAMPLES]
held_graphs  = all_graphs[TRAIN_SAMPLES:]
print(f"\nParsed {len(all_graphs)} graphs  ({errors} errors skipped)")
print(f"Train: {len(train_graphs)}  |  Held-out: {len(held_graphs)}")

# ── Induce lexicon ──────────────────────────────────────────────────────────
print("\nInducing MTLG lexicon...")
inducer = MtlgInducer(LANGUAGE)
lexicon = inducer.induce_from_stream(iter(train_graphs), max_trees=len(train_graphs))
print(f"Lexicon: {len(lexicon.entries)} lemmas")

# ── Bootstrap TRDs ──────────────────────────────────────────────────────────
print("Bootstrapping TRDs...")
bootstrapper = TrdBootstrapper(n_clusters=N_TRD_CLUSTERS)
trds = bootstrapper.bootstrap(train_graphs, LANGUAGE)
print(f"TRDs: {len(trds)} clusters")

# ── Evaluate on held-out ────────────────────────────────────────────────────
print("\nEvaluating on held-out sentences...")
covered = 0
trd_assigned = 0
for g in held_graphs:
    lemma_hits = sum(1 for n in g.nodes if lexicon.best_type(n.lemma) is not None)
    if len(g.nodes) > 0 and lemma_hits / len(g.nodes) >= 0.5:
        covered += 1
    if bootstrapper._model and bootstrapper.vocab:
        vec = graph_to_modal_vector(g, bootstrapper.vocab)
        try:
            bootstrapper._model.predict(vec)
            trd_assigned += 1
        except Exception:
            pass

n_h = len(held_graphs) or 1
print(f"Lexicon coverage (≥50% nodes): {covered}/{n_h}  ({100*covered/n_h:.1f}%)")
print(f"TRD assignment success        : {trd_assigned}/{n_h}  ({100*trd_assigned/n_h:.1f}%)")

## Step 9 — Integration test suite

Runs the same pipeline as `lcs/induction/test_pipeline.py` inline so results appear in the notebook.

In [ ]:
import traceback

TEST_SENTENCES = [
    "The scientist discovered a new particle in the laboratory.",
    "She quickly opened the window and looked outside.",
    "The committee approved the proposed regulation unanimously.",
    "A dog chased the cat across the yard.",
    "He believes that honesty is the best policy.",
]

print("=" * 60)
print("INTEGRATION TEST SUITE")
print("=" * 60)

passed = 0
failed = 0

for sent in TEST_SENTENCES:
    try:
        tokens   = preprocess_for_type_assignment(sent, LANGUAGE)
        clean    = " ".join(t.split("[")[0] for t in tokens)
        tree     = parser.parse(clean)
        g        = ud_tree_to_mtlg(tree)

        assert len(g.nodes) > 0, "Graph has no nodes"

        # Lexicon lookup
        for node in g.nodes:
            _ = lexicon.best_type(node.lemma)  # should not raise

        # TRD assignment
        if bootstrapper._model and bootstrapper.vocab:
            vec = graph_to_modal_vector(g, bootstrapper.vocab)
            bootstrapper._model.predict(vec)

        print(f"  PASS  {sent[:60]}")
        passed += 1
    except Exception as e:
        print(f"  FAIL  {sent[:60]}")
        print(f"        {e}")
        failed += 1

print("=" * 60)
print(f"Results: {passed} passed, {failed} failed  ({'OK' if failed == 0 else 'FAILURES'})")
print("=" * 60)

## Step 10 — Visualise modal mode and UCCA category distributions

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

mode_counts = Counter()
ucca_counts = Counter()

for g in train_graphs:
    for n in g.nodes:
        mode_counts[n.modal_mode] += 1
        ucca_counts[n.category_id] += 1

UCCA_NAMES = {0:"Scene",1:"Process",2:"Connector",3:"Ground",4:"Adverbial",5:"State",6:"Participant"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Modal mode
modes, m_vals = zip(*sorted(mode_counts.items())) if mode_counts else ([],[]) 
axes[0].bar(modes, m_vals, color=["#4C72B0","#DD8452","#55A868"][:len(modes)])
axes[0].set_title("Modal Mode Distribution")
axes[0].set_ylabel("Count")

# UCCA categories
ucca_labels = [UCCA_NAMES.get(k, str(k)) for k in sorted(ucca_counts)]
ucca_vals   = [ucca_counts[k] for k in sorted(ucca_counts)]
axes[1].bar(ucca_labels, ucca_vals, color="#8172B3")
axes[1].set_title("UCCA Category Distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

## Step 11 — Inference demo

In [ ]:
DEMO_SENTENCE = "The scientist discovered a new particle in the laboratory."

tokens = preprocess_for_type_assignment(DEMO_SENTENCE, LANGUAGE)
clean  = " ".join(t.split("[")[0] for t in tokens)
tree   = parser.parse(clean)
g      = ud_tree_to_mtlg(tree)

print(f"Sentence : {DEMO_SENTENCE}")
print(f"Nodes    : {len(g.nodes)}")
print(f"Edges    : {len(g.edges)}")
print()

# Lexicon lookup per node
print(f"{'Token':<14} {'Lemma':<14} {'Mode':<10} {'UCCA':<12} {'Arity'}")
print("-" * 56)
for node in g.nodes:
    entry    = lexicon.best_type(node.lemma)
    lex_mode = entry.modal_mode if entry else "—"
    lex_cat  = entry.ucca_cat   if entry else "—"
    print(f"{node.text:<14} {node.lemma:<14} {lex_mode:<10} {str(lex_cat):<12} {node.arity}")

# TRD assignment
trd_label = "unknown"
if bootstrapper._model and bootstrapper.vocab:
    vec    = graph_to_modal_vector(g, bootstrapper.vocab)
    trd_id = bootstrapper._model.predict(vec)
    for t in bootstrapper.trds:
        if t.trd_id == trd_id:
            trd_label = t.label
            break

print(f"\nAssigned TRD cluster: {trd_label}")

## Step 12 — Render MTLG dependency graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

UCCA_COLORS = {0:"#55A868",1:"#DD8452",2:"#937860",3:"#DA8BC3",4:"#8172B3",5:"#C44E52",6:"#4C72B0"}
UCCA_NAMES  = {0:"Scene",1:"Process",2:"Connector",3:"Ground",4:"Adverbial",5:"State",6:"Participant"}
MODE_STYLE  = {"diamond":"solid","box":"dashed","lozenge":"dotted"}
MODE_COLOR  = {"diamond":"#4C72B0","box":"#DD8452","lozenge":"#55A868"}
MODE_SYM    = {"diamond":"◇","box":"□","lozenge":"◊"}

G = nx.DiGraph()
for node in g.nodes:
    G.add_node(node.token_id, label=node.text, ucca=node.category_id)
for edge in g.edges:
    G.add_edge(edge.src_id, edge.dst_id, mode=edge.modal_mode, deprel=edge.deprel)

node_colors = [UCCA_COLORS.get(G.nodes[n].get("ucca", 0), "#aaaaaa") for n in G.nodes]
pos = nx.spring_layout(G, seed=42, k=2.2)

fig, ax = plt.subplots(figsize=(14, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000, ax=ax, alpha=0.95)
nx.draw_networkx_labels(G, pos,
    labels={n: G.nodes[n].get("label", str(n)) for n in G.nodes},
    font_size=8, font_color="white", font_weight="bold", ax=ax)

for (u, v, data) in G.edges(data=True):
    mode = data.get("mode", "diamond")
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
        style=MODE_STYLE.get(mode, "solid"),
        edge_color=MODE_COLOR.get(mode, "#888"),
        arrows=True, arrowsize=20, width=2, ax=ax,
        connectionstyle="arc3,rad=0.1")

edge_labels = {(e.src_id, e.dst_id): e.deprel for e in g.edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

legend_patches = [mpatches.Patch(color=c, label=UCCA_NAMES[k]) for k, c in UCCA_COLORS.items()]
mode_lines = [
    plt.Line2D([0],[0], color=MODE_COLOR[m], lw=2, linestyle=MODE_STYLE[m], label=f"{MODE_SYM[m]} {m}")
    for m in ("diamond","box","lozenge")
]
ax.legend(handles=legend_patches + mode_lines, loc="lower left", fontsize=8,
          title="UCCA / Mode", ncol=2)
ax.set_title(f'MTLG Graph: "{DEMO_SENTENCE}"', fontsize=11, fontweight="bold")
ax.axis("off")
plt.show()

## Step 13 — Download trained artefacts

In [ ]:
import json
from google.colab import files

# Serialise lexicon
lexicon_path = f"/content/{LANGUAGE}_lexicon.json"
with open(lexicon_path, "w") as f:
    json.dump({e.lemma: {"mode": e.modal_mode, "ucca": e.ucca_cat, "count": e.count}
               for e in lexicon.entries.values()}, f, indent=2)

# Serialise TRDs
trds_path = f"/content/{LANGUAGE}_trds.json"
with open(trds_path, "w") as f:
    json.dump([{"id": t.trd_id, "label": t.label, "patterns": t.infon_patterns}
               for t in bootstrapper.trds], f, indent=2)

print(f"Saved: {lexicon_path}")
print(f"Saved: {trds_path}")

files.download(lexicon_path)
files.download(trds_path)

## Step 14 — Launch Streamlit UI (public ngrok URL)

The Streamlit app (`app.py`) provides:
- **Train tab** — stream C4, induce lexicon, bootstrap TRDs with live progress bars
- **Inference & Visualization tab** — analyse sentences and render MTLG graphs interactively

Set your free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# ── Configure ngrok ─────────────────────────────────────────────────────────
NGROK_AUTH_TOKEN = ""  # paste your token here (or set env var NGROK_AUTHTOKEN)

import os
token = NGROK_AUTH_TOKEN or os.environ.get("NGROK_AUTHTOKEN", "")
if token:
    ngrok.set_auth_token(token)
else:
    print("WARNING: no ngrok token set — tunnel may fail on free tier.")

# ── Launch Streamlit in background ──────────────────────────────────────────
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", f"{REPO_DIR}/app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)  # give Streamlit time to start

# ── Open ngrok tunnel ───────────────────────────────────────────────────────
public_url = ngrok.connect(8501)
print("\n" + "=" * 60)
print(f"  Streamlit UI  →  {public_url}")
print("=" * 60)
print("Open the URL above to use the interactive Train / Inference UI.")
print("Run the cell below to stop the server when finished.")

In [ ]:
# ── Stop the Streamlit server and close the tunnel ──────────────────────────
ngrok.disconnect(public_url)
streamlit_proc.terminate()
print("Server stopped.")